# EN Transformer NER Training on Colab GPU

## 前提条件
1. **ランタイム → ランタイムのタイプを変更 → T4 GPU → 保存**
2. 上から順にすべてのセルを実行
3. **Step 6 で重みがローカルにダウンロードされるまで完了ではない**

## 注意事項
- `spacy[cuda12x]` は **絶対に使わない**（Colabのcupyを壊す）
- `spacy init config --gpu` で公式GPU configを生成（手書きNG）
- Colabランタイム切断でファイルが消えるため、**必ず Step 5-6 を実行**

## 前回の結果 (test set)
| Entity | F1 | CNN比 |
|---|---|---|
| Overall | **96.56%** | +1.3pt |
| PERSON | 96.18% | +0.4 |
| ORGANIZATION | 95.71% | +1.4 |
| ADDRESS | 99.20% | +1.2 |
| BANK_ACCOUNT | **97.64%** | **+6.1** |
| DATE_OF_BIRTH | 92.49% | +0.8 |

In [ ]:
# ============================================================
# Step 1: Install & verify GPU
# ============================================================
!pip install -q spacy spacy-transformers
!nvidia-smi
import spacy, thinc.util
print(f'spacy={spacy.__version__}, cupy={thinc.util.has_cupy}')
assert thinc.util.has_cupy, '\n\n*** CuPy not available! ***\nランタイム → ランタイムのタイプを変更 → T4 GPU → 保存\n'

In [ ]:
# ============================================================
# Step 2: Clone repo & generate GPU config
# ============================================================
!git clone https://github.com/plenoai/pleno-anonymize.git 2>&1 | tail -3
%cd /content/pleno-anonymize/packages/training

# spacy init config --gpu が正しいGPU config を生成する（手書きではダメ）
!python -m spacy init config --lang en --pipeline transformer,ner --gpu /tmp/base.cfg
!python -m spacy init fill-config /tmp/base.cfg configs/gpu.cfg
print('\n✅ GPU config generated')

In [ ]:
# ============================================================
# Step 3: Fetch training data & train
# ============================================================
!git fetch origin tmp/en-data 2>&1 | tail -1
!git checkout origin/tmp/en-data -- data/processed/en/
!echo "Training data:" && ls -lh data/processed/en/

!python -m spacy train configs/gpu.cfg \
    --output output/en-transformer \
    --paths.train data/processed/en/train.spacy \
    --paths.dev data/processed/en/dev.spacy \
    --gpu-id 0

In [ ]:
# ============================================================
# Step 4: Evaluate on test set
# ============================================================
!python -m spacy evaluate output/en-transformer/model-best data/processed/en/test.spacy \
    --gpu-id 0 \
    --output output/en-transformer/test_scores.json

import json
with open('output/en-transformer/test_scores.json') as f:
    scores = json.load(f)
print(f"\n{'='*50}")
print(f"Overall F1: {scores['ents_f']*100:.2f}%")
print(f"{'='*50}")
for entity, s in scores['ents_per_type'].items():
    print(f"  {entity:20s} F1={s['f']*100:.2f}%  P={s['p']*100:.2f}%  R={s['r']*100:.2f}%")

In [ ]:
# ============================================================
# Step 5: Save model to Google Drive (ランタイム切断対策)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE_DIR = '/content/drive/MyDrive/pleno-models/en_transformer_best'

# 前回の保存があれば削除
if os.path.exists(DRIVE_DIR):
    shutil.rmtree(DRIVE_DIR)

# model-best をコピー
shutil.copytree('output/en-transformer/model-best', DRIVE_DIR)

# test_scores.json もコピー
shutil.copy2('output/en-transformer/test_scores.json', f'{DRIVE_DIR}/test_scores.json')

# 確認
!du -sh {DRIVE_DIR}
!ls {DRIVE_DIR}/
print(f'\n✅ Model saved to Google Drive: {DRIVE_DIR}')

In [ ]:
# ============================================================
# Step 6: Download model-best as tar.gz (ローカル保存用)
# ⚠️ このステップを必ず実行してローカルにダウンロードすること!
# ============================================================
import tarfile
from google.colab import files

TAR_PATH = '/content/en_transformer_model_best.tar.gz'

# tar.gz 作成
with tarfile.open(TAR_PATH, 'w:gz') as tar:
    tar.add('output/en-transformer/model-best', arcname='model-best')
    tar.add('output/en-transformer/test_scores.json', arcname='test_scores.json')

!ls -lh {TAR_PATH}
print('\n⬇️  Downloading to your local machine...')
print('   Save to: packages/models/en_ner_en_transformer-0.1.0/')
print('   Then run: git add && git commit && git push')
files.download(TAR_PATH)

## ローカルでの保存手順

ダウンロードした `en_transformer_model_best.tar.gz` を展開してリポジトリに保存:

```bash
cd packages/models
mkdir -p en_ner_en_transformer-0.1.0
tar xzf ~/Downloads/en_transformer_model_best.tar.gz -C en_ner_en_transformer-0.1.0/
git add en_ner_en_transformer-0.1.0/
git commit -m 'feat: add EN transformer NER model (F1=96.56%)'
git push
```

※ `.gitattributes` で LFS tracking が設定済み（`packages/models/**/model` 等）